# Modèle de recalage en python avec $SO(2)$

### Approche Hamiltonienne - Tir géodésique
Pour résoudre ce problème numériquement, on utilise un algorithme itératif :
- **Initialisation**:  On choisit un moment initial, par exemple $p_0^{(0)}=0$.
- **Boucle d'optimisation** (pour chaque itération $k$) :
    1. _Tir géodésique_ ([Méthode d'Euler explicite](https://fr.wikipedia.org/wiki/M%C3%A9thode_d%27Euler#Euler_explicite)) :
        En partant de $x_0=x_S$ et $p_0=p_0^{(k)}$, on intègre simultanément les deux dynamiques :
        $$
        x_{t+\Delta t}=x_t+\Delta t\,A^*_t\,x_t,\qquad p_{t+\Delta t}=p_t+\Delta t\,A^*_t\,p_t,
        $$
        où le champ optimal $A^*_t$ est recalculé à chaque pas à partir de $(x_t,p_t)$ courants. On obtient ainsi $x_1=x_1(p_0^{(k)})$ la déformation finale.
    2. _Évaluation de l’énergie_ : On calcule le coût global $J(p_0^{(k)})$.
    3. _Mise à jour_ ([Descente de gradient](https://fr.wikipedia.org/wiki/Algorithme_du_gradient) sur $p_0$) : On calcule
    $$
    p_0^{(k+1)}=p_0^{(k)}-\alpha\,\nabla J(p_0^{(k)})
    $$
La descente de gradient peut être fait avec l'algorithme [L-BFGS](https://docs.pytorch.org/docs/2.12/generated/torch.optim.LBFGS.html) de pytorch.

In [ ]:
import torch
from torch.autograd import grad
import matplotlib.pyplot as plt
import numpy as np

## Fonctions pour le recalage rigide $SO(2)$

On souhaite toujours minimiser le problème suivant :
$$
\begin{aligned}
\min_{v\in L^2([0,1],V)}~&J(v)=\frac{1}{2}\int_{0}^{1}\|v_t\|_{V}^2\,dt+\|I_0\circ\varphi_1^{-1}-I_1\|\\
&\text{tel que }\left\{\begin{array}{ll}\dot{\varphi}_t=v_t\circ\varphi_t\\\varphi_0=Id\end{array}\right.
\end{aligned}
$$

Et on retrouve ce qu'on avait dans le chapitre "**Formulation Hamiltonienne du recalage sur $SO(n)$**" :
$$
\begin{aligned}
\min_{p_0}~&\frac{1}{2}\int_{0}^{1}\|A_t\|_{V}^2\,dt+\|q_1-q_T\|\\
\text{tel que les équa}&\text{tions hamiltoniennes sont satisfaites.}
\end{aligned}
$$

`skew_matrix` construit la matrice anti-symétrique $A(\omega)$ pour $SO(2)$.

In [ ]:
def skew_matrix(omega):
    A = torch.zeros((2, 2), dtype=omega.dtype, device=omega.device)
    A[0, 1] = -omega.squeeze()
    A[1, 0] = omega.squeeze()
    return A

### Étape 1 : _Calcul du générateur optimal_
`hamiltonian_A` calcule $A^*_t=\frac12\left(p_t q_t^{\top}-q_t p_t^{\top}\right)$ à partir des tenseurs $p_t$ et $x_t$.

In [ ]:
def hamiltonian_A(p, q):
    return 0.5 * (p.T @ q - q.T @ p)

### Étape 2 : _Tir géodésique_
`Euler_Hamilton` ntègre simultanément $x_t$ et $p_t$ par la méthode d'Euler explicite sur $t\in[0,1]$, en réévaluant $A^*_t$ à chaque pas.

In [ ]:
def Euler_Hamilton(q0, p0, nt):
    dt = 1.0 / (nt - 1)
    q = q0.clone()
    p = p0.clone()

    Q = torch.zeros((q0.shape[0], q0.shape[1], nt), dtype=q0.dtype, device=q0.device)
    P = torch.zeros((p0.shape[0], p0.shape[1], nt), dtype=p0.dtype, device=p0.device)
    Q[:, :, 0] = q
    P[:, :, 0] = p

    for t in range(1, nt):
        A = hamiltonian_A(p, q)
        q = q + dt * (q @ A.T)
        p = p + dt * (p @ A.T)
        Q[:, :, t] = q
        P[:, :, t] = p

    return q, p, Q, P

### Étape 3 : _Évaluation de l’énergie_

`hamiltonian_loss` calcule l'énergie totale $J(p_0) =\frac{1}{2}{\|p_0\|}^2+$ `dataloss_fnc(q_finale)`

In [ ]:
def hamiltonian_loss(dataloss_fnc, nt):
    def loss(q0, p0):
        q_final, p_final, _, _ = Euler_Hamilton(q0, p0, nt)
        dloss = dataloss_fnc(q_final)
        reg = 0.5 * (p0 ** 2).sum()
        return reg + dloss
    return loss

### Étape 4 : _Mise à jour (Descente de gradient)_

`opti_hamiltonian` optimise le moment initial $p_0$ par descente de gradient.

In [ ]:
def opti_hamiltonian(q0, loss_fnc, n_iter, eta):
    p0 = torch.zeros_like(q0, requires_grad=True)
    optimizer = torch.optim.LBFGS([p0], lr=eta, line_search_fn='strong_wolfe')
    losses = []

    def closure():
        optimizer.zero_grad()
        L = loss_fnc(q0, p0)
        L.backward()
        return L

    for i in range(n_iter):
        L = optimizer.step(closure)
        losses.append(L.item())

        print(f"Iteration {i+1:3d} - Loss: {L.item():.6f} | |p0|: {p0.norm().item():.4f}")
        print(f"")

    return p0.detach(), losses

`GaussLinKernel` et `lossVarifoldCurve` :

In [ ]:
def GaussLinKernel(sigma):
    '''
    Define "Gaussian-CauchyBinet" kernel :math:`(K(x,y,u,v)b)_i = \sum_j \exp(-\gamma\|x_i-y_j\|^2) \langle u_i,v_j \rangle^2 b_j`
    '''
    def fun(x, y, u, v, b):
        gamma = 1 / (sigma * sigma)
        D2 = ((x[:,None,:]-y[None,:,:])**2).sum(dim=2)
        K = (-D2 * gamma).exp() * (u[:,None,:] * v[None,:,:]).sum(dim=2) ** 2
        return K @ b
    return fun

def lossVarifoldCurve(FS, VT, FT, K):
    '''
    Fonction qui calcule la loss varifold entre une forme source et une forme cible.
    '''
    
    def get_center_length_tangents(F, V):
        V0, V1 = (
            V.index_select(0, F[:, 0]),
            V.index_select(0, F[:, 1]),
        )
        centers, tangents = .5*(V0+V1), (V1-V0)
        non_zero_rows = torch.any(tangents != 0, dim=1)
        centers, tangents = centers[non_zero_rows], tangents[non_zero_rows]
        length = (tangents**2).sum(dim=1)[:, None].sqrt()
        return centers, length, tangents/length

    CT, LT, TTn = get_center_length_tangents(FT, VT)
    cst = (LT * K(CT, CT, TTn, TTn, LT)).sum()

    def loss(VS):
        CS, LS, TSn = get_center_length_tangents(FS, VS)
        return (
            cst
            + (LS * K(CS, CS, TSn, TSn, LS)).sum()
            - 2 * (LS * K(CS, CT, TSn, TTn, LT)).sum()
        )

    return loss

## Configurations de la forme et optimisation de la rotation :

In [ ]:
# torch type and device
use_cuda = torch.cuda.is_available()
torchdeviceId = torch.device("cuda:0") if use_cuda else "cpu"
torchdtype = torch.float64

n_s = 10  # Nombre de points source
n_t = 10  # Nombre de points target
time_shoot = 20  # Temps de shooting

#####################################################################
### Definition des points source et cible

# Source
VS = torch.cat((torch.zeros(n_s, 1), torch.linspace(0, 4, n_s).reshape(n_s, 1)), 1)
FS = torch.cat((torch.arange(0, n_s - 1).reshape(n_s - 1, 1), torch.arange(1, n_s).reshape(n_s - 1, 1)), dim=1)

# Cible
th = torch.linspace(0, 0.4 * torch.pi, n_t).reshape(n_t, 1) # Ajuste pour la rotation
VT = torch.cat((2.0 * torch.sin(th), torch.linspace(0, 4, n_t).reshape(n_t, 1)), 1)
FT = torch.cat((torch.arange(0, n_t - 1).reshape(n_t - 1, 1), torch.arange(1, n_t).reshape(n_t - 1, 1)), dim=1)

#####################################################################
# Transfert sur le Device

q0 = VS.clone().detach().to(dtype=torchdtype, device=torchdeviceId)
VT = VT.clone().detach().to(dtype=torchdtype, device=torchdeviceId)
FS = FS.clone().detach().to(dtype=torch.long, device=torchdeviceId)
FT = FT.clone().detach().to(dtype=torch.long, device=torchdeviceId)

#####################################################################
# Noyaux et Attache aux donnees

sigma = torch.tensor([1.5], dtype=torchdtype, device=torchdeviceId)
K_lin = GaussLinKernel(sigma) # Noyau pour l'espace des Varifolds

dataloss_varifold = lossVarifoldCurve(FS, VT, FT, K_lin)
loss_fnc = hamiltonian_loss(dataloss_varifold, time_shoot)

# Optimisation (moment initial p0, Approche Hamiltonienne / tir geodesique)
print("Debut de l'optimisation Hamiltonienne du recalage SO(2)...")
p0_opt, loss_history = opti_hamiltonian(q0, loss_fnc, n_iter=100, eta=1e-1)

# Trajectoire finale
q_final, p_final, Q_traj, P_traj = Euler_Hamilton(q0, p0_opt, time_shoot)

## Affichage de la rotation (la même qu'avec l'autre méthode):

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

q0_np = q0.cpu().numpy()
q_final_np = q_final.cpu().numpy()
VT_np = VT.cpu().numpy()

# Plot Source
ax.plot(q0_np[:, 0], q0_np[:, 1], 'go-', label='Source (Initiale)', alpha=0.5)

# Plot Target
ax.plot(VT_np[:, 0], VT_np[:, 1], 'bx-', label='Target (Cible)', lw=2)

# Plot Final
ax.plot(q_final_np[:, 0], q_final_np[:, 1], 'ro-', label='Rotated SO(2)', lw=2)

ax.axis('equal')
ax.legend()
ax.set_title(rf"Recalage SO(2) termine (Hamiltonien, $\|p_0\|$ optimise = {p0_opt.norm().item():.4f})")
plt.grid()
plt.show()

### Rappel de l'approche directe avec un scalaire (pour comparaison)

On reintroduit ici les fonctions `Euler`, `rotation_loss` et `opti_rotation` (approche avec le scalaire $\omega$), afin de pouvoir les comparer directement a l'approche hamiltonienne.

In [ ]:
def Euler(q0, omega, nt):
    dt = 1.0 / (nt - 1)
    q = q0.clone()
    A = skew_matrix(omega)
    Q = torch.zeros((q0.shape[0], q0.shape[1], nt), dtype=q0.dtype, device=q0.device)
    Q[:, :, 0] = q

    for t in range(1, nt):
        q = q + dt * (q @ A.T)
        Q[:, :, t] = q

    return q, Q

def rotation_loss(dataloss_fnc, nt):
    def loss(q0, omega):
        q_final, _ = Euler(q0, omega, nt)
        dloss = dataloss_fnc(q_final)
        reg = 0.5 * (omega ** 2)
        return 0.1 * reg + dloss
    return loss


def opti_rotation(q0, loss_fnc, n_iter, eta, omega_init):
    omega = torch.tensor([omega_init], dtype=q0.dtype, device=q0.device, requires_grad=True)
    optimizer = torch.optim.LBFGS([omega], lr=eta, line_search_fn='strong_wolfe')
    losses = []

    def closure():
        optimizer.zero_grad()
        L = loss_fnc(q0, omega)
        L.backward()
        return L

    for i in range(n_iter):
        L = optimizer.step(closure)
        losses.append(L.item())

    return omega.detach(), losses


## Comparaison des temps d'execution : Hamiltonien vs Scalaire

On compare ici le temps d'optimisation entre l'approche Hamiltonienne (`opti_hamiltonian`, moment $p_0$) et l'approche scalaire (`opti_rotation`, angle $\omega$), a nombre d'iterations egal.

In [ ]:
import time

n_iter_comp = 100

# --- Approche Hamiltonienne ---
t0 = time.time()
_ = opti_hamiltonian(q0, loss_fnc, n_iter=n_iter_comp, eta=1e-1)
t_hamilton = time.time() - t0

# --- Approche Scalaire ---
loss_fnc_scalar = rotation_loss(dataloss_varifold, time_shoot)
t0 = time.time()
_ = opti_rotation(q0, loss_fnc_scalar, n_iter=n_iter_comp, eta=1e-1, omega_init=0.0)
t_scalar = time.time() - t0

print(f"Temps Hamiltonien ({n_iter_comp} iter) : {t_hamilton:.4f} s")
print(f"Temps Scalaire    ({n_iter_comp} iter) : {t_scalar:.4f} s")
print(f"Rapport (Hamiltonien / Scalaire)       : {t_hamilton/t_scalar:.2f}")

plt.figure(figsize=(4, 4))
plt.bar(["Hamiltonien", "Scalaire"], [t_hamilton, t_scalar], color=["tab:red", "tab:blue"])
plt.ylabel("Temps (s)")
plt.title("Temps d'execution de l'optimisation")
plt.show()
